# L71 · 🎨 ASR 流水线图形化 — 波形 → 声谱图 → token → 文字路径可视化

**目标**：用图直观理解 CTC 多路径对齐、Whisper 解码器的 cross-attention 热力图、以及 WER 误差的结构。

🔗 **Aurora 连接**：
- `aurora.audio.mel` — 将原始波形转换为 mel 频谱，送入 Whisper encoder
- Whisper encoder 对每帧输出隐状态向量
- Whisper decoder 的每个输出 token 通过 cross-attention 对这些隐状态加权求和，本节可视化该权重矩阵

## 本节核心直觉

ASR 系统的输出是离散 token 序列，但输入是连续音频帧——这两个序列的长度完全不同，对齐它们是核心难题。
CTC 通过引入 blank 符号并对所有合法路径求和，让模型无需人工对齐即可训练。
Attention 机制则在解码时动态决定“当前输出 token 最该关注哪些音频帧”，热力图让这一隐式对齐一目了然。

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

## 1. CTC 对齐：多路径 → 同一目标序列

CTC（Connectionist Temporal Classification）在每个时间步输出一个字符（含特殊 blank 符号 `-`）。
解码规则：先去掉连续重复，再去掉所有 blank，得到最终序列。

对于目标词 `"HI"`，以下路径全部合法：

```
路径长度 T=5 示例（vocab: H I -）
  H H I - I   -> 去重 -> H I I -> 去blank -> HI  check
  H - I - -   -> 去重 -> H - I - -> 去blank -> HI check
  - H - I -   -> 去重 -> - H - I - -> 去blank -> HI check
  H - - I I   -> 去重 -> H - I -> 去blank -> HI check
```

CTC loss = `-log( 所有合法路径的概率之和 )`，前向算法在格（lattice）上高效求和。
下图用颜色标出所有合法路径经过的节点，帮助你感受“合法路径数”随 T 增长如何爆炸式增加。

In [ ]:
# CTC 合法路径可视化 (静态 lattice 快照)
# 目标序列: 'HI', 时间步 T=6, 词表: {H:0, I:1, -:2(blank)}
# 枚举所有长度为 T 的路径，筛选出解码结果为 'HI' 的合法路径

VOCAB = ['H', 'I', '-']
TARGET = ['H', 'I']
T = 6
V = len(VOCAB)


def ctc_decode(path):
    # CTC decode: 去连续重复 -> 去blank
    collapsed = [path[0]] + [path[i] for i in range(1, len(path)) if path[i] != path[i-1]]
    return [c for c in collapsed if c != '-']


from itertools import product

all_paths = list(product(range(V), repeat=T))
valid_paths = [p for p in all_paths if ctc_decode([VOCAB[c] for c in p]) == TARGET]
print(f'总路径数: {V}^{T} = {V**T}')
print(f'合法路径数 (目标="HI"): {len(valid_paths)}')
print(f'前5条合法路径: {[tuple(VOCAB[c] for c in p) for p in valid_paths[:5]]}')

# 绘制 lattice: 节点 (t, v)，合法路径经过的节点用红色标出
node_visited = np.zeros((T, V), dtype=int)
for p in valid_paths:
    for t, v in enumerate(p):
        node_visited[t, v] += 1

fig, ax = plt.subplots(figsize=(9, 3))
for t in range(T):
    for v in range(V):
        count = node_visited[t, v]
        color = plt.cm.Reds(min(count / len(valid_paths) * 3, 1.0)) if count > 0 else '#e0e0e0'
        ax.add_patch(plt.Circle((t, v), 0.35, color=color, zorder=2))
        ax.text(t, v, VOCAB[v], ha='center', va='center', fontsize=10, zorder=3,
                color='white' if count > 0 else '#aaa')

ax.set_xlim(-0.6, T - 0.4)
ax.set_ylim(-0.6, V - 0.4)
ax.set_xticks(range(T))
ax.set_xticklabels([f't={i}' for i in range(T)])
ax.set_yticks(range(V))
ax.set_yticklabels(VOCAB)
ax.set_title(f'CTC Lattice (目标="HI", T={T})\n红色深浅 = 该节点被合法路径经过的频次')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()
print(f'合法路径占比: {len(valid_paths)/V**T*100:.1f}%')

## 2. Cross-Attention 热力图：decoder token 关注哪些音频帧

Whisper 的 decoder 在每个自回归步骤生成一个 token 时，对 encoder 输出做 cross-attention：

```
score(q, k) = q . k / sqrt(d_k)
alpha = softmax(score)          # shape: (n_tokens, n_frames)
context = alpha @ encoder_out   # 加权聚合
```

`alpha` 矩阵就是热力图的来源——行 = 输出 token，列 = 音频帧，颜色深浅 = 注意力权重。
对齐良好的模型，热力图沿对角线高亮：先发的音素对应早期帧，后发的对应晚期帧。
偏离对角线意味着跳帧、重复或幻觉。

In [ ]:
# 模拟 Cross-Attention 热力图 (合成数据)
# 实际使用时替换为 Whisper hook 捕获的真实 attention 权重

rng = np.random.default_rng(42)
N_TOKENS = 10   # 输出 token 数
N_FRAMES = 40   # encoder 帧数 (mel 帧，约 25ms/帧)

token_labels = list('HELLO WORLD')  # 10个字符作为 token 标签
attn = np.zeros((N_TOKENS, N_FRAMES))
for i in range(N_TOKENS):
    center = int(i / N_TOKENS * N_FRAMES)
    spread = 3
    for j in range(N_FRAMES):
        attn[i, j] = np.exp(-0.5 * ((j - center) / spread) ** 2)
    attn[i] += rng.uniform(0, 0.05, N_FRAMES)
    attn[i] /= attn[i].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 左图: 良好对齐 (对角线)
im0 = axes[0].imshow(attn, aspect='auto', cmap='viridis', origin='upper')
axes[0].set_xlabel('编码器帧 (audio frames)')
axes[0].set_ylabel('输出 token')
axes[0].set_yticks(range(N_TOKENS))
axes[0].set_yticklabels(list(token_labels))
axes[0].set_title('Cross-Attention 热力图\n(对角线 = 对齐良好)')
plt.colorbar(im0, ax=axes[0], label='注意力权重')

# 右图: 扰乱对齐 (模拟幻觉/跳帧)
attn_bad = attn.copy()
for i in [3, 4, 7]:
    attn_bad[i] = rng.dirichlet(np.ones(N_FRAMES) * 0.3)
im1 = axes[1].imshow(attn_bad, aspect='auto', cmap='viridis', origin='upper')
axes[1].set_xlabel('编码器帧 (audio frames)')
axes[1].set_ylabel('输出 token')
axes[1].set_yticks(range(N_TOKENS))
axes[1].set_yticklabels(list(token_labels))
axes[1].set_title('Cross-Attention 热力图\n(部分 token 注意力散乱 = 幻觉风险)')
plt.colorbar(im1, ax=axes[1], label='注意力权重')

plt.tight_layout()
plt.show()
print('左图对角线清晰；右图 token 3/4/7 注意力弥散，对应幻觉或跳帧现象')

## 参数实验：调整注意力扩散程度，观察热力图变化

控制参数：
- `spread`（高斯宽度）：值越大，注意力越扩散；`spread=1` 近乎 hard alignment，`spread=8` 对应模型不确定
- `noise_level`：注意力背景噪声，模拟低置信度帧

**预期现象**：
- `spread` 从 1 → 8：对角线变宽 → 模型对当前 token 对应帧的定位越来越模糊
- `noise_level > 0.15`：背景噪声抬高，热力图整体变灰，对齐结构消失
- 比较不同 `spread` 下每行熵 `H = -sum(alpha * log(alpha))` 的变化

In [ ]:
# 参数实验: spread & noise_level
rng2 = np.random.default_rng(0)
N_TOKENS2, N_FRAMES2 = 8, 32

def make_attn(spread, noise_level):
    a = np.zeros((N_TOKENS2, N_FRAMES2))
    for i in range(N_TOKENS2):
        center = int(i / N_TOKENS2 * N_FRAMES2)
        for j in range(N_FRAMES2):
            a[i, j] = np.exp(-0.5 * ((j - center) / spread) ** 2)
        a[i] += rng2.uniform(0, noise_level, N_FRAMES2)
        a[i] /= a[i].sum()
    return a

configs = [
    (1,   0.01,  'spread=1, noise=0.01  (hard, 清晰)'),
    (4,   0.05,  'spread=4, noise=0.05  (中等)'),
    (8,   0.15,  'spread=8, noise=0.15  (散乱, 幻觉风险)'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (sp, nl, title) in zip(axes, configs):
    a = make_attn(sp, nl)
    im = ax.imshow(a, aspect='auto', cmap='plasma', origin='upper')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('音频帧')
    ax.set_ylabel('输出 token')
    entropies = -np.sum(a * np.log(a + 1e-9), axis=1)
    ax.text(0.98, 0.02, f'平均熵={entropies.mean():.2f}',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, color='white')
    plt.colorbar(im, ax=ax)

plt.suptitle('参数实验: 注意力扩散程度对热力图的影响', y=1.02)
plt.tight_layout()
plt.show()

for sp, nl, title in configs:
    a = make_attn(sp, nl)
    H = -np.sum(a * np.log(a + 1e-9), axis=1).mean()
    print(f'spread={sp:2d}, noise={nl:.2f} -> 平均注意力熵 = {H:.3f}')
print('熵越高说明 decoder 越不确定当前 token 对应哪些帧，是幻觉的预警信号')

## 本节收束

本节通过 `ctc_decode` 枚举了所有合法 CTC 路径，并用 lattice 热力图展示了“多路径 → 同一目标”的核心对齐思想。
`alpha = softmax(q·k / sqrt(d_k))` 矩阵的可视化揭示了 Whisper decoder cross-attention 的隐式对齐结构，对角线清晰度是幻觉预警的直观指标。
两张图都直接对应 `aurora.audio.mel` → encoder → decoder 的推理管道：mel 帧是横轴，token 序列是纵轴。
下一节将实现 WER 计算（编辑距离分解为替换/插入/删除），并结合本节的对齐图定位具体误差来源。